# 閾値（しきい値）トリガによるデータ取得 — DMA + Plotly 版

このノートブックは `detector-acquisition.ipynb` の **後半（Option B：全波形ロギング）** を **DMA (Deep Memory Acquisition / AXI)** で書き直し、プロットには numpy 依存が緩い **Plotly** を採用したものです。

- **Option A** (CSV ピーク・積分ロギング) → **そのまま** 標準 ACQ (`rp_AcqGetOldestDataV`) を使います。
- **Option B** (HDF5 全波形ロギング) → **DMA** (`rp_AcqAxi*`) に置き換え。データを Linux カーネルバッファを経由せず DDR に直接書き込むので、長いバッファでも取りこぼしが起きにくくなります。

**注意:**
Red Pitaya の高速アナログ入力の電圧レンジは、本体上の **入力ジャンパ** の位置で決まります。

- **HV** 設定 → 入力レンジ ±20 V
- **LV** 設定 → 入力レンジ ±1 V

詳しくは公式ドキュメントの[こちらの章](https://redpitaya.readthedocs.io/en/latest/developerGuide/hardware/125-14/fastIO.html#analog-inputs)をご参照ください。

下の図のように、**高速アナログ出力 (OUT) と高速アナログ入力 (IN) を SMA ケーブルで直結（ループバック接続）** してください。ジャンパは **±1 V (LV)** に設定されていることを必ず確認してください。

![Fast loop back](../img/FastIOLoopBack.png "ループバック接続の例")

## DMA とは何か

Deep Memory Acquisition (DMA) は、Red Pitaya の **DDR メモリに直接書き込む** タイプのデータ取得です。標準の `rp_AcqGetOldestData*` 系 API との違いは:

- バッファ長が **16384 サンプル固定ではない** (任意のサンプル数。64 バイト境界に揃える必要あり)
- AXI 経由で DDR に直接ストリーミング (Linux 側の DMA 予約領域を共有)
- フルレート 125 MSPS で取得可能
- 取り出しに時間がかかる (バッファが大きいほど顕著)
- **標準 ACQ と並行して動作可能** ← 本ノートブックの Option A と Option B を切り替えて使えるのはこのおかげ

本ノートブックでは **1 ショット = 16384 サンプル/ch (オリジナルと同じ長さ)** で DMA を構成します。チャンネルあたり 32 KB、合計 64 KB なので、Linux 側の DDR 圧迫はほぼゼロです。詳細は[公式ドキュメント](https://redpitaya.readthedocs.io/en/latest/appsFeatures/remoteControl/deepMemoryAcquisition.html)を参照してください。

## 0. Plotly のインストール (最初に 1 回だけ実行)

Red Pitaya のシステム Python (3.10) で動作確認済みのバージョン組み合わせです。`numpy` を巻き込んで再ビルドが走らないように、すべて **バイナリ wheel が配布済み** のバージョンに固定し、`--no-deps` で純 Python 依存だけ別途入れます。

下のセルを **1 回だけ** 実行 → カーネルを再起動してください。インストール済みなら 2 回目以降はスキップしてかまいません。

```bash
# シェルで実行する場合 (Red Pitaya に SSH した状態で)
pip install --user --no-deps "plotly==5.24.1"
pip install --user "tenacity>=8.2.0,<9" "packaging>=24.0"
```

ポイント:
- `--no-deps` を付けると、Plotly のサブ依存が新しい numpy を要求して再ビルドが走るのを防げます。
- `--user` を付けると `~/.local/` 以下に入るため、システムの site-packages を汚しません。
- `tenacity`, `packaging` は Plotly 5.24 が動作するために必要な純 Python の依存だけ別途入れています。

In [ ]:
# ノートブックから直接インストールしたい場合はこのセル (1 回だけ)。既に入っている場合はスキップ。
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--no-deps", "plotly==5.24.1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "tenacity>=8.2.0,<9", "packaging>=24.0"])
print("Plotly インストール完了。カーネルを一度再起動してから次のセルへ進んでください。")

## ライブラリと FPGA イメージの読み込み

データのプロットに **Plotly**、配列演算に `numpy` を使います。Plotly は numpy への依存が緩いので、システムの numpy が古くても新しくても動きます。Red Pitaya 固有の `rp_overlay`（FPGA イメージ読み込み）と `rp`（API 本体）も読み込みます。

In [ ]:
import time
import datetime
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import importlib, paper_style
importlib.reload(paper_style)  # pick up edits to paper_style.py without kernel restart
from rp_overlay import overlay
import rp
fpga = overlay()
rp.rp_Init()
print("FPGA ready, plotly", __import__('plotly').__version__)

## マクロ一覧

標準 ACQ で使うマクロに加え、DMA では以下も登場します。

- **デシメーション (Decimation)** — `RP_DEC_1`, `RP_DEC_2`, `RP_DEC_4`, ... `RP_DEC_65536` (DMA は 1〜65536 の任意整数も可)
- **取得トリガ (Acquisition trigger)** — `RP_TRIG_SRC_DISABLED`, `RP_TRIG_SRC_NOW`, `RP_TRIG_SRC_CHA_PE`, `RP_TRIG_SRC_CHA_NE`, `RP_TRIG_SRC_CHB_PE`, `RP_TRIG_SRC_CHB_NE`, `RP_TRIG_SRC_EXT_PE`, `RP_TRIG_SRC_EXT_NE`, `RP_TRIG_SRC_AWG_PE`, `RP_TRIG_SRC_AWG_NE`
- **トリガ状態 (Acquisition trigger state)** — `RP_TRIG_STATE_TRIGGERED`, `RP_TRIG_STATE_WAITING`
- **バッファサイズ (Buffer size)** — `ADC_BUFFER_SIZE`, `DAC_BUFFER_SIZE`
- **高速アナログチャンネル (Fast analog channels)** — `RP_CH_1`, `RP_CH_2`

**DMA 専用 API (一部)**
- `rp_AcqAxiGetMemoryRegion()` → `[status, start_addr, size_samples]`
- `rp_AcqAxiSetDecimationFactor(dec)`
- `rp_AcqAxiSetTriggerDelay(ch, samples_after_trigger)`
- `rp_AcqAxiSetBufferSamples(ch, addr, samples)`
- `rp_AcqAxiEnable(ch, bool)`
- `rp_AcqAxiGetBufferFillState(ch)` / `rp_AcqAxiGetWritePointerAtTrig(ch)`
- `rp_AcqAxiGetDataRaw(ch, pos, size, buf)` ← データ読み出し (RAW, int16)

## 用語の説明

データ取得（アクイジション）を正しく設定するためには、いくつかの専門用語を知っておく必要があります。

- **トリガモーメント (Triggering moment)** — トリガ条件が満たされ、装置がデータの取り込みを開始する瞬間です。
- **トリガ条件 (Trigger condition)** — 「トリガレベル」と「シグナルフロント」の組み合わせで決まります。
- **トリガレベル (Trigger level)** — 入力信号がこの電圧 [V] に達したらデータ取得を始める、という閾値の値です。
- **シグナルフロント (Signal front)** — **立ち上がりエッジ／立ち下がりエッジ (positive/negative edge)** とも呼ばれます。
- **デシメーション (Decimation, 間引き)** — 何サンプルおきに 1 サンプル保存するかの設定です。
- **平均化 (Averaging)** — デシメーションが 2 以上のとき、保存される 1 サンプルは「捨てる側のサンプルすべての平均」になります。
- **取得単位 (Acquisition units)** — *RAW*（ADC の生値）と *Volts*（電圧 [V]）。**DMA は RAW のみ** で、本ノートブックでは ±1 V (LV) 換算を手動で行います (`raw / 8192`)。
- **トリガディレイ (Trigger delay)** — 標準 ACQ ではバッファ内のトリガ位置をシフトする値、**DMA では「トリガ後に何サンプル取り続けるか」** を意味します (意味が異なるので注意)。

## トリガレベルの設定 (標準 ACQ + 信号発生器)

ループバックで動作確認するための信号発生器設定と、Option A・確認プロットで使う標準 ACQ 用の `acqADC()` を定義します。

In [ ]:
# === 信号発生器のパラメータ ===
channel_list = [rp.RP_CH_1, rp.RP_CH_2]  # 出力チャンネルの一覧
trigger_list = [rp.RP_T_CH_1, rp.RP_T_CH_2]  # トリガチャンネルの一覧
# 取得用トリガソースの一覧。index 0 -> CH1 (CHA_PE), index 1 -> CH2 (CHB_PE)
acq_trig_sour_list = [rp.RP_TRIG_SRC_CHA_PE, rp.RP_TRIG_SRC_CHB_PE]

# === 取得 (Acquisition) パラメータ ===
dec = rp.RP_DEC_1                # デシメーション設定 (1 -> 125 MS/s, そのまま保存)
N = 16384                        # 取得バッファのサンプル数 (固定)
trigger_ch = 0                   # トリガに使うチャンネルのインデックス (0 -> CH1, 1 -> CH2)
trig_lvl = 0.10                  # トリガレベル [V]
trig_dly = int(1/2*N) - 30       # 標準 ACQ 用トリガディレイ [samples]: トリガ瞬間をバッファ先頭から 30 サンプル目に置く
duty = int(1/64*N)               # Option A で実際に使う波形長 [samples] (バッファ先頭から duty サンプルだけ抜き出す)
rp.rp_AcqReset()

# DIO0_N からトリガ信号を出力する設定
rp.rp_SetDpinEnableTrigOutput(True)
rp.rp_SetSourceTrigOutput(rp.OUT_TR_ADC)

##### 標準 ACQ 設定 (Option A と確認プロット用) #####
rp.rp_AcqSetDecimation(dec)
rp.rp_AcqSetTriggerLevel(trigger_list[trigger_ch], trig_lvl)
rp.rp_AcqSetTriggerDelay(trig_dly)

# トリガ／バッファ充填待ちのループでビジーウェイトしないためのスリープ間隔 [秒]
_POLL_SLEEP_S = 1e-4

def acqADC(trigger_src):
    """標準 ACQ で 1 ショット取得し、両チャンネルの波形 (numpy 配列, 長さ duty, V) を返す。"""
    rp.rp_AcqStart()
    time.sleep(0.001)
    rp.rp_AcqSetTriggerSrc(trigger_src)
    rp.rp_GenTriggerOnly(channel_list[trigger_ch])  # 信号発生器をトリガ

    while rp.rp_AcqGetTriggerState()[1] != rp.RP_TRIG_STATE_TRIGGERED:
        time.sleep(_POLL_SLEEP_S)
    while not rp.rp_AcqGetBufferFillState()[1]:
        time.sleep(_POLL_SLEEP_S)

    waveform_arr = []
    for channel in channel_list:
        fbuff = rp.fBuffer(N)
        rp.rp_AcqGetOldestDataV(channel, N, fbuff)
        data = np.fromiter(fbuff, dtype=np.float32, count=duty)
        waveform_arr.append(data)

    return waveform_arr


print("標準 ACQ パラメータ設定と acqADC 定義が完了しました。")

## 信号のプロット (Plotly + 標準 ACQ で動作確認)

`acqADC()` でトリガが意図通り効いているか軽く確認します。Plotly はインタラクティブで、ホバーで値が読めたり、ドラッグでズームできたりします。

- **トリガを待たずに即時取得** したい場合: `rp.RP_TRIG_SRC_NOW`
- **トリガを使って取得** する場合: `acq_trig_sour_list[trigger_ch]`

In [ ]:
# 1 ショット取り、2 チャンネル分の波形を Plotly で重ねてプロット
# (paper_style: 16:9, Okabe-Ito 配色, 内向き目盛り, 4 辺枠 が自動適用)
w, h = paper_style.figsize()  # 1280 x 720 = 16:9
waveform_arr = acqADC(acq_trig_sour_list[trigger_ch])
fig = go.Figure()
for i, waveform in enumerate(waveform_arr):
    fig.add_trace(go.Scatter(y=waveform, mode='lines', name=f'ch{i}'))
fig.update_layout(
    xaxis_title='Sample',
    yaxis_title='Voltage [V]',
    width=w, height=h,
)
paper_style.show(fig)


## DMA セットアップと取得関数 (Option B 用)

ここからが本ノートブックの追加部分です。`acqADC_DMA()` は `acqADC()` と同じ「トリガソースを渡して、両チャンネルの numpy 配列リストを受け取る」インタフェースですが、内部では DMA (`rp_AcqAxi*`) を使います。

**振る舞いの違い:**

| 項目 | `acqADC()` 標準 | `acqADC_DMA()` DMA |
|---|---|---|
| バッファ長 | 16384 固定、先頭 `duty=256` を抜く | 16384 全サンプルを返す |
| 取得単位 | V (API が変換) | RAW を `1/8192` で V 換算 (LV ジャンパ前提) |
| データ書き込み先 | カーネルバッファ経由 | **DDR に直接** |
| トリガ位置 | バッファ先頭から 30 サンプル目 | バッファ先頭 (post-trigger 16384 サンプル) |

DMA は **HV (±20 V) ジャンパ** を使う場合、換算係数を `20.0 / 8192` に変える必要があります。`_DMA_LSB_VOLTS` を書き換えてください。

In [ ]:
# === DMA パラメータ ===
DMA_DATA_SIZE = N                    # 1 ショットのサンプル数 (16384、64 バイト境界 OK)
DMA_TRIG_DELAY = DMA_DATA_SIZE       # トリガ後に取り続けるサンプル数 (= バッファ全長)
_DMA_LSB_VOLTS = 1.0 / 8192.0        # 14-bit 符号付き ADC: ±1V (LV) なら 1/8192。±20V (HV) なら 20/8192

# 読み出しバッファをショット間で使い回す (毎回確保しない)
_dma_ibuf = [rp.i16Buffer(DMA_DATA_SIZE), rp.i16Buffer(DMA_DATA_SIZE)]


def setup_dma():
    """DMA を構成して両チャンネルで有効化する。Option B のループ前に 1 度だけ呼ぶ。"""
    region = rp.rp_AcqAxiGetMemoryRegion()
    axi_start = region[1]            # 予約領域先頭アドレス
    axi_size  = region[2]            # 予約領域サイズ (samples)
    print(f"DMA region: start=0x{axi_start:x} size_samples=0x{axi_size:x}")

    rp.rp_AcqAxiSetDecimationFactor(dec)

    # トリガ後に DMA_DATA_SIZE サンプル取り続ける = ちょうどバッファを埋めて止まる
    rp.rp_AcqAxiSetTriggerDelay(rp.RP_CH_1, DMA_TRIG_DELAY)
    rp.rp_AcqAxiSetTriggerDelay(rp.RP_CH_2, DMA_TRIG_DELAY)

    # 予約領域を半分ずつ CH1/CH2 に割り当てる
    rp.rp_AcqAxiSetBufferSamples(rp.RP_CH_1, axi_start, DMA_DATA_SIZE)
    rp.rp_AcqAxiSetBufferSamples(rp.RP_CH_2, axi_start + axi_size // 2, DMA_DATA_SIZE)

    rp.rp_AcqAxiEnable(rp.RP_CH_1, True)
    rp.rp_AcqAxiEnable(rp.RP_CH_2, True)

    # トリガレベルは標準 ACQ と共有 (rp_AcqSetTriggerLevel)
    rp.rp_AcqSetTriggerLevel(trigger_list[trigger_ch], trig_lvl)
    print("DMA enabled on CH1, CH2")


def teardown_dma():
    """DMA を無効化。Option B のループ終了時に必ず呼ぶ (例外時も finally で)。"""
    rp.rp_AcqAxiEnable(rp.RP_CH_1, False)
    rp.rp_AcqAxiEnable(rp.RP_CH_2, False)


def acqADC_DMA(trigger_src):
    """DMA で 1 ショット取得し、両チャンネルの波形 (numpy float32, 長さ DMA_DATA_SIZE, V) を返す。"""
    rp.rp_AcqStart()
    time.sleep(0.001)
    rp.rp_AcqSetTriggerSrc(trigger_src)
    rp.rp_GenTriggerOnly(channel_list[trigger_ch])  # 信号発生器をトリガ (loopback テスト用)

    # トリガ条件待ち
    while rp.rp_AcqGetTriggerState()[1] != rp.RP_TRIG_STATE_TRIGGERED:
        time.sleep(_POLL_SLEEP_S)

    # DMA バッファ充填待ち (チャンネル別に確認)
    while not rp.rp_AcqAxiGetBufferFillState(rp.RP_CH_1)[1]:
        time.sleep(_POLL_SLEEP_S)
    while not rp.rp_AcqAxiGetBufferFillState(rp.RP_CH_2)[1]:
        time.sleep(_POLL_SLEEP_S)

    rp.rp_AcqStop()

    # トリガ瞬間の書き込みポインタ (= 取得波形の先頭サンプル位置)
    pos1 = rp.rp_AcqAxiGetWritePointerAtTrig(rp.RP_CH_1)[1]
    pos2 = rp.rp_AcqAxiGetWritePointerAtTrig(rp.RP_CH_2)[1]

    rp.rp_AcqAxiGetDataRaw(rp.RP_CH_1, pos1, DMA_DATA_SIZE, _dma_ibuf[0].cast())
    rp.rp_AcqAxiGetDataRaw(rp.RP_CH_2, pos2, DMA_DATA_SIZE, _dma_ibuf[1].cast())

    ch1 = (np.fromiter(_dma_ibuf[0], dtype=np.int16, count=DMA_DATA_SIZE)
             .astype(np.float32) * _DMA_LSB_VOLTS)
    ch2 = (np.fromiter(_dma_ibuf[1], dtype=np.int16, count=DMA_DATA_SIZE)
             .astype(np.float32) * _DMA_LSB_VOLTS)
    return [ch1, ch2]


print("DMA セットアップと acqADC_DMA 定義が完了しました。")

### DMA 動作確認 (任意、1 ショットだけ取って Plotly で表示)

Option B のループに入る前に、DMA が正しく構成できているか単発で確認します。CH1/CH2 を縦に並べて表示します。

In [ ]:
setup_dma()
try:
    waveform_arr = acqADC_DMA(acq_trig_sour_list[trigger_ch])
    w, h = paper_style.figsize(rows=2)  # 16:9 per panel, 2 行積み = 1280 x 1440
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('ch0 (DMA)', 'ch1 (DMA)'),
                        vertical_spacing=0.08)
    for i, waveform in enumerate(waveform_arr):
        # 全 16384 サンプルを描くと重いので先頭 duty サンプルだけ表示 (ホバーで全体ズーム可)
        fig.add_trace(go.Scatter(y=waveform[:duty], mode='lines', name=f'ch{i}'),
                      row=i + 1, col=1)
    fig.update_xaxes(title_text='Sample', row=2, col=1)
    fig.update_yaxes(title_text='Voltage [V]')
    fig.update_layout(width=w, height=h, showlegend=False)
    paper_style.show(fig)
    print(f'shape: ch1={waveform_arr[0].shape} ch2={waveform_arr[1].shape}')
    print(f'range ch1: [{waveform_arr[0].min():.4f}, {waveform_arr[0].max():.4f}] V')
finally:
    teardown_dma()


## 計測 (Measurement)

ここからが本番のデータ取得です。データは `./data/` ディレクトリに保存されます (無ければ自動作成)。

**ご注意:** 次の 2 つの取得セル (Option A / Option B) は **どちらか一方を選んで実行** してください。

- **Option A** — ピーク値・積分値ロギング (CSV, `.dat`)。**標準 ACQ** を使用。後段集計が軽くて済みます。
- **Option B** — 全波形ロギング (HDF5, `.h5`)。**DMA** を使用。`DMA_DATA_SIZE = 16384` サンプル/ch を毎ショット保存。

Option A → Option B の順で連続実行する場合、Option B 側で自動的に `setup_dma()` / `teardown_dma()` を呼ぶので追加の切替操作は不要です。

In [ ]:
# === Option A: ピーク値・積分値ロギング (CSV) — 標準 ACQ ===
import json
from pathlib import Path

DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

start_date = datetime.datetime.now()
filename = start_date.strftime('%Y-%m-%d-%H%M%S')
out_path = DATA_DIR / f'{filename}.dat'
meta_path = DATA_DIR / f'{filename}.json'
print(f"計測開始: {out_path}")

meta = {
    'start_datetime': start_date.isoformat(),
    'mode': 'option_a_peak_sum_csv',
    'acq_path': 'standard_acq',
    'decimation_macro': 'RP_DEC_1',
    'samples_per_shot': N,
    'duty': duty,
    'trigger_ch_index': trigger_ch,
    'trigger_src': 'RP_TRIG_SRC_CHA_PE' if trigger_ch == 0 else 'RP_TRIG_SRC_CHB_PE',
    'trigger_level_V': trig_lvl,
    'trigger_delay_samples': trig_dly,
    'input_range_jumper': 'LV (+/-1V)',
    'csv_columns': ['totaltime', 'deadtime', 'sum_ch1', 'max_ch1', 'sum_ch2', 'max_ch2'],
}
meta_path.write_text(json.dumps(meta, indent=2))

total_deadtime = 0.0
start_time = time.time()

with out_path.open('w') as f:
    f.write(','.join(meta['csv_columns']) + '\n')
    try:
        while True:
            waveform_arr = acqADC(acq_trig_sour_list[trigger_ch])
            acq_end_time = time.time()
            totaltime = acq_end_time - start_time
            ch1 = waveform_arr[0]
            ch2 = waveform_arr[1]
            data = [
                totaltime,
                total_deadtime,
                float(np.sum(ch1)),
                float(np.max(ch1)),
                float(np.sum(ch2)),
                float(np.max(ch2)),
            ]
            f.write(",".join(map(str, data)) + '\n')
            total_deadtime += time.time() - acq_end_time

    except KeyboardInterrupt:
        print("ユーザによって停止されました。")

### Option B — 全波形ロギング (HDF5) — **DMA**

このセルは Option A の代替です。各ショットで取得した 2 チャンネル分の波形を **DMA で取得し、HDF5 ファイルに追記** します。1 ショットあたり `DMA_DATA_SIZE = 16384` サンプル/ch を **そのまま** 保存します (オリジナル版で発生していた `duty (256) → N (16384)` の shape 不整合バグはここで解消)。

**Option A と両方を実行する必要はありません。** どちらか好きな方を使ってください。HDF5 出力には `h5py` が必要です。

```bash
pip install --user h5py
```

In [ ]:
# === Option B: 全波形ロギング (HDF5) — DMA ===
import h5py
from pathlib import Path

DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

start_date = datetime.datetime.now()
filename = start_date.strftime('%Y-%m-%d-%H%M%S')
out_path = DATA_DIR / f'{filename}.h5'
print(f"計測開始 (DMA): {out_path}")
total_deadtime = 0.0
start_time = time.time()

setup_dma()

try:
    with h5py.File(out_path, 'w') as h5:
        # 計測条件を HDF5 ファイル属性として埋め込み
        h5.attrs['start_datetime']        = start_date.isoformat()
        h5.attrs['mode']                  = 'option_b_full_waveform_hdf5_dma'
        h5.attrs['acq_path']              = 'dma_axi'
        h5.attrs['decimation_macro']      = 'RP_DEC_1'
        h5.attrs['samples_per_shot']      = DMA_DATA_SIZE
        h5.attrs['dma_trigger_delay']     = DMA_TRIG_DELAY
        h5.attrs['trigger_ch_index']      = trigger_ch
        h5.attrs['trigger_src']           = 'RP_TRIG_SRC_CHA_PE' if trigger_ch == 0 else 'RP_TRIG_SRC_CHB_PE'
        h5.attrs['trigger_level_V']       = trig_lvl
        h5.attrs['input_range_jumper']    = 'LV (+/-1V)'
        h5.attrs['lsb_to_volts']          = _DMA_LSB_VOLTS

        # 各データセットを「行 0、可変長」で初期化。あとで resize して 1 行ずつ append する
        dset_t   = h5.create_dataset('totaltime', shape=(0,), maxshape=(None,),
                                      dtype='f8', chunks=(64,))
        dset_dt  = h5.create_dataset('deadtime',  shape=(0,), maxshape=(None,),
                                      dtype='f8', chunks=(64,))
        dset_ch1 = h5.create_dataset('ch1', shape=(0, DMA_DATA_SIZE), maxshape=(None, DMA_DATA_SIZE),
                                      dtype='f4', chunks=(1, DMA_DATA_SIZE), compression='lzf')
        dset_ch2 = h5.create_dataset('ch2', shape=(0, DMA_DATA_SIZE), maxshape=(None, DMA_DATA_SIZE),
                                      dtype='f4', chunks=(1, DMA_DATA_SIZE), compression='lzf')

        n_shots = 0
        try:
            while True:
                waveform_arr = acqADC_DMA(acq_trig_sour_list[trigger_ch])
                acq_end_time = time.time()
                totaltime = acq_end_time - start_time

                dset_t.resize(n_shots + 1, axis=0);   dset_t[n_shots]   = totaltime
                dset_dt.resize(n_shots + 1, axis=0);  dset_dt[n_shots]  = total_deadtime
                dset_ch1.resize(n_shots + 1, axis=0); dset_ch1[n_shots] = waveform_arr[0]
                dset_ch2.resize(n_shots + 1, axis=0); dset_ch2[n_shots] = waveform_arr[1]
                n_shots += 1

                total_deadtime += time.time() - acq_end_time

        except KeyboardInterrupt:
            print(f"ユーザによって停止されました。{n_shots} ショットを保存しました。")
finally:
    teardown_dma()
    print("DMA を無効化しました。")

## クリーンアップ

ノートブック全体を終了する前に、念のため DMA を無効化し、リソースを解放しておきます。

(Option B が正常終了/`KeyboardInterrupt` した場合は既に `teardown_dma()` が走っていますが、二重に呼んでも害はありません。)

In [ ]:
try:
    teardown_dma()
except Exception as e:
    print(f"teardown_dma() スキップ: {e}")
rp.rp_Release()
print("終了しました。")

## トラブルシューティング

### `fig.show()` を実行しても何も出ない
JupyterLab の場合、Plotly 5.x は追加 extension なしで描画できます。それでも空白の場合は以下を試してください。

```python
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # クラシック Notebook 向け
# または
pio.renderers.default = "jupyterlab"           # JupyterLab 向け
```

### それでも描画されない / オフラインで使いたい
HTML として保存してブラウザで開く方法が確実です。

```python
fig.write_html('shot.html')   # ./shot.html を生成。ブラウザで開く
```

### `pip install` 中に numpy がビルドされ始める
Plotly のサブ依存 (`narwhals` など) が新しい numpy を要求して再ビルドが走ることがあります。冒頭で書いたように **必ず `--no-deps` を付けて plotly 本体を入れ、足りない純 Python 依存だけ別途入れてください**。

### DMA で `Already enabled` エラーが出る
前回の実行で `teardown_dma()` が呼ばれずに残っているケースです。クリーンアップセル (一番下) を一度実行してから setup し直してください。